# Diffusion Tensor Imaging (DTI)

DTI is the most widely used diffusion model. It approximates the 3-D diffusion profile as an **ellipsoid** — a simple but powerful description of how water diffuses in each voxel.

## The model

The signal equation for DTI is:

$$S(b, \hat{g}) = S_0 \, e^{-b \, \hat{g}^T \mathbf{D} \hat{g}}$$

where $\mathbf{D}$ is the **3×3 diffusion tensor** (symmetric, positive definite), $\hat{g}$ is the gradient direction, and $b$ is the b-value.

Fitting the tensor requires at least **6 non-collinear gradient directions** (plus b=0). HCP data has 90 directions — much more than needed, giving very stable estimates.

## Derived scalar maps

After fitting, the tensor's eigenvalues $\lambda_1 \geq \lambda_2 \geq \lambda_3$ yield clinically useful maps:

| Map | Formula | Meaning |
|---|---|---|
| FA | $\sqrt{\frac{3}{2} \frac{\sum(\lambda_i - \bar{\lambda})^2}{\sum \lambda_i^2}}$ | 0=isotropic, 1=perfectly anisotropic |
| MD | $(\lambda_1 + \lambda_2 + \lambda_3)/3$ | Overall diffusivity |
| AD | $\lambda_1$ | Diffusivity along the principal axis |
| RD | $(\lambda_2 + \lambda_3)/2$ | Diffusivity perpendicular to fibre |

## Limitation

DTI **cannot resolve crossing fibres**. A voxel with two fibre populations at 90° appears isotropic (low FA). This is why CSD was developed (next chapter).

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import fsl_dtifit, plot_metric_map

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
dti_dir  = Path('../../data/hcp/100307/dti')
dti_dir.mkdir(parents=True, exist_ok=True)

# Use eddy-corrected data if available; fall back to raw
dwi_nii  = str(prep_dir / 'dwi_eddy.nii.gz') if (prep_dir / 'dwi_eddy.nii.gz').exists() \
            else str(data_dir / 'data.nii.gz')
bvals_f  = str(data_dir / 'bvals')
bvecs_f  = str(data_dir / 'bvecs')
# Use rotated bvecs if eddy was run
rotated  = str(prep_dir / 'dwi_eddy.eddy_rotated_bvecs')
if Path(rotated).exists():
    bvecs_f = rotated
    print('Using rotated bvecs from FSL eddy')
mask     = str(data_dir / 'nodif_brain_mask.nii.gz')

print(f'DWI: {dwi_nii}')

## Approach A: FSL dtifit

In [ ]:
# ─── [FSL] dtifit ─────────────────────────────────────────────────────────────
#
# Fits the tensor using weighted least squares.
#
# Outputs (all named <base>_*.nii.gz):
#   FA, MD, L1, L2, L3   — scalar maps
#   V1, V2, V3           — eigenvectors (V1 = principal direction)
#   MO                   — mode of anisotropy
#   S0                   — mean b=0 signal
#   tensor               — the full tensor (6-volume image)

fsl_dti_base = str(dti_dir / 'fsl_dti')

fsl_cmd = [
    'dtifit',
    '--data='  + dwi_nii,
    '--mask='  + mask,
    '--bvecs=' + bvecs_f,
    '--bvals=' + bvals_f,
    '--out='   + fsl_dti_base,
    '--wls',    # weighted least squares (more accurate than OLS)
    '--save_tensor',
]

print('FSL dtifit command:')
print(' '.join(fsl_cmd))
print()

result = subprocess.run(fsl_cmd, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ FSL dtifit complete')
    import os
    fsl_files = list(dti_dir.glob('fsl_dti_*.nii.gz'))
    print(f'  Output files: {[f.name for f in fsl_files]}')

## Approach B: MRtrix3 dwi2tensor + tensor2metric

In [ ]:
# ─── [MRtrix3] dwi2tensor ─────────────────────────────────────────────────────
#
# Step 1: Fit the tensor
# -mask: only compute within brain
# -b0: output the estimated b=0 image

mrt_tensor  = str(dti_dir / 'mrt_tensor.mif')

mrt_cmd1 = [
    'dwi2tensor',
    str(prep_dir / 'dwi_eddy.mif') if (prep_dir / 'dwi_eddy.mif').exists() else str(data_dir / 'data.nii.gz'),
    mrt_tensor,
    '-mask', mask,
    '-force',
]
print('MRtrix3 dwi2tensor command:')
print(' '.join(mrt_cmd1))

result = subprocess.run(mrt_cmd1, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# ─── [MRtrix3] tensor2metric ──────────────────────────────────────────────────
#
# Step 2: Extract scalar maps from the tensor
# One call extracts all metrics at once.

mrt_fa  = str(dti_dir / 'mrt_FA.nii.gz')
mrt_md  = str(dti_dir / 'mrt_MD.nii.gz')
mrt_ad  = str(dti_dir / 'mrt_AD.nii.gz')
mrt_rd  = str(dti_dir / 'mrt_RD.nii.gz')
mrt_v1  = str(dti_dir / 'mrt_V1.nii.gz')

mrt_cmd2 = [
    'tensor2metric',
    mrt_tensor,
    '-fa',  mrt_fa,
    '-md',  mrt_md,
    '-ad',  mrt_ad,
    '-rd',  mrt_rd,
    '-vector', mrt_v1,   # principal eigenvector
    '-mask', mask,
    '-force',
]
print('MRtrix3 tensor2metric command:')
print(' '.join(mrt_cmd2))

result = subprocess.run(mrt_cmd2, capture_output=True, text=True)
print(result.stdout or '(done)')
if result.returncode != 0:
    print('STDERR:', result.stderr)
else:
    print('✓ MRtrix3 tensor metrics extracted')

## Approach C: DIPY TensorModel

In [ ]:
# ─── [DIPY] TensorModel ──────────────────────────────────────────────────────
#
# DIPY fits the tensor in Python. Use only b≤1000 volumes for DTI.
# Higher b-values violate the monoexponential model assumption.

from dipy.io.gradients import read_bvals_bvecs
from dipy.core.gradients import gradient_table
from dipy.reconst.dti import TensorModel

print('Loading data ...')
img   = nib.load(dwi_nii)
data  = img.get_fdata()
maskd = nib.load(mask).get_fdata().astype(bool)

bvals, bvecs = read_bvals_bvecs(bvals_f, bvecs_f)
gtab = gradient_table(bvals, bvecs)

# Select single-shell b=1000 + b=0 for DTI
sel = (bvals < 50) | ((bvals > 900) & (bvals < 1100))
data_dti = data[..., sel]
gtab_dti = gradient_table(bvals[sel], bvecs[sel])

print(f'Using {sel.sum()} volumes (b=0 + b≈1000) for DTI fitting')

# Fit the model
tenmodel = TensorModel(gtab_dti, fit_method='WLS')   # WLS = weighted LS
tenfit   = tenmodel.fit(data_dti, mask=maskd)

print('✓ Tensor fit complete')

# Extract metrics
from dipy.reconst.dti import fractional_anisotropy, mean_diffusivity

FA_dipy = fractional_anisotropy(tenfit.evals)
MD_dipy = mean_diffusivity(tenfit.evals)
V1_dipy = tenfit.evecs[..., 0]    # principal eigenvector

# Save
nib.save(nib.Nifti1Image(FA_dipy.astype(np.float32), img.affine),
         str(dti_dir / 'dipy_FA.nii.gz'))
nib.save(nib.Nifti1Image(MD_dipy.astype(np.float32), img.affine),
         str(dti_dir / 'dipy_MD.nii.gz'))

print(f'FA range: {FA_dipy[maskd].min():.3f} – {FA_dipy[maskd].max():.3f}')
print(f'MD mean (brain): {MD_dipy[maskd].mean():.6f} mm²/s')

## Compare outputs: all three tools

In [ ]:
# Compare FA maps from all three tools
fsl_fa_path  = str(dti_dir / 'fsl_dti_FA.nii.gz')
mrt_fa_path  = mrt_fa
dipy_fa_path = str(dti_dir / 'dipy_FA.nii.gz')

sources = []
for path, label in [(fsl_fa_path, 'FSL dtifit FA'),
                    (mrt_fa_path, 'MRtrix3 FA'),
                    (dipy_fa_path, 'DIPY FA')]:
    if Path(path).exists():
        sources.append((path, label))

z = data.shape[2] // 2
fig, axes = plt.subplots(1, len(sources), figsize=(5 * len(sources), 5))
if len(sources) == 1:
    axes = [axes]

for ax, (path, label) in zip(axes, sources):
    fa_vol = nib.load(path).get_fdata()
    ax.imshow(fa_vol[:, :, z].T, cmap='hot', vmin=0, vmax=1, origin='lower')
    ax.set_title(label, fontsize=11)
    ax.axis('off')

fig.suptitle('FA maps — side-by-side comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('All three tools should produce nearly identical FA maps.')
print('Small differences arise from:')
print('  • Different shell selection (DIPY used b=1000 only; FSL/MRtrix3 used all shells)')
print('  • Slight differences in the fitting algorithm implementation')

In [ ]:
# Quantitative comparison: correlation between FA maps
if len(sources) == 3:
    fa_fsl  = nib.load(fsl_fa_path).get_fdata()[maskd]
    fa_mrt  = nib.load(mrt_fa_path).get_fdata()[maskd]
    fa_dipy = FA_dipy[maskd]

    r_fsl_mrt  = np.corrcoef(fa_fsl, fa_mrt)[0, 1]
    r_fsl_dipy = np.corrcoef(fa_fsl, fa_dipy)[0, 1]
    r_mrt_dipy = np.corrcoef(fa_mrt, fa_dipy)[0, 1]

    print('Voxel-wise FA correlation (within brain mask):')
    print(f'  FSL  vs MRtrix3 : r = {r_fsl_mrt:.4f}')
    print(f'  FSL  vs DIPY    : r = {r_fsl_dipy:.4f}')
    print(f'  MRtrix3 vs DIPY : r = {r_mrt_dipy:.4f}')
    print()
    print('r > 0.99 = tools are effectively equivalent')
    print('r < 0.95 = investigate the difference before proceeding')

## Visualise the fibre orientation (V1 map)

The RGB colour convention: R=left-right, G=anterior-posterior, B=superior-inferior.

In [ ]:
# ─── [DIPY] RGB directional map ───────────────────────────────────────────────
fa_3d = FA_dipy

# Scale V1 by FA so only high-FA (organised) voxels show strong colour
rgb = np.abs(V1_dipy) * fa_3d[..., np.newaxis]
rgb = np.clip(rgb, 0, 1)

z = rgb.shape[2] // 2
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(rgb[:, :, z].transpose(1, 0, 2), origin='lower')
ax.set_title('DIPY: FA-weighted RGB map (V1 principal eigenvector)\n'
             'Red=L-R  Green=A-P  Blue=S-I', fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.show()

---

## Summary

| Aspect | FSL dtifit | MRtrix3 dwi2tensor | DIPY TensorModel |
|---|---|---|---|
| Speed | Fast | Fast | Moderate (Python) |
| Output format | NIfTI | .mif → NIfTI | NumPy arrays → NIfTI |
| Shell selection | All shells (not ideal) | All shells | You control it |
| Fitting method | WLS (--wls flag) | WLS | WLS / OLS / NLLS |
| Result | Same | Same | Same |
| Best for | Quick maps, group studies | MRtrix3 pipeline | Research, custom analysis |

> **Recommendation**: For DTI specifically, use DIPY or FSL with only b=1000 volumes. For CSD (next), you need MRtrix3 or DIPY.

**Next**: [Constrained Spherical Deconvolution →](02_csd.ipynb)